In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import json
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import Dataset
from transformers import AutoTokenizer
from sklearn.metrics import precision_recall_fscore_support
from huggingface_hub import login, hf_hub_download
import zipfile









In [ ]:
# ==========================================
# 1. AUTHENTICATION & DATA LOADING
# ==========================================
login(token=os.environ["HF_TOKEN"])

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_TOKENIZER_REPO = "A-240242/legal-modern-decoder-checkpoints"

def load_cjpe_file(filename):
    path = hf_hub_download(
        repo_id=HF_DATASET_REPO, 
        filename=f"Phase_4_CJPE/{filename}", 
        repo_type="dataset", 
        token=HF_TOKEN
    )
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

print("Downloading Phase 4 CJPE Data...")
train_data = load_cjpe_file("Train/train.json")
dev_data = load_cjpe_file("Train/dev.json")

DEBUG_MODE = False
if DEBUG_MODE:
    train_data = train_data[:100]
    dev_data = dev_data[:20]


Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

In [ ]:
# ==========================================
# 2. TOKENIZER & DUAL-ANCHOR CHUNKING
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(HF_TOKENIZER_REPO, token=os.environ["HF_TOKEN"])
tokenizer.padding_side = "right"

def prepare_dataset_records(raw_records, tokenizer, max_length=1024):
    input_ids_list = []
    attention_mask_list = []
    labels_list = []
    
    for record in raw_records:
        doc_text = record.get("text", "")
        label = record.get("label", 0)
        
        tokenized = tokenizer(doc_text, truncation=False, return_attention_mask=False)
        full_ids = tokenized["input_ids"]
        
        if len(full_ids) <= max_length:
            chunks = [full_ids]
        else:
            first_chunk = full_ids[:max_length]
            last_chunk = full_ids[-max_length:]
            chunks = [first_chunk, last_chunk]
            
        for chunk in chunks:
            input_ids_list.append(chunk)
            attention_mask_list.append([1] * len(chunk))
            labels_list.append(label)
            
    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list,
        "label": labels_list
    }

print("Processing Dataset Features...")
train_features = prepare_dataset_records(train_data, tokenizer)
dev_features = prepare_dataset_records(dev_data, tokenizer)

train_dataset = Dataset.from_dict(train_features)
dev_dataset = Dataset.from_dict(dev_features)

def custom_collate_fn(batch):
    input_ids = [torch.tensor(item["input_ids"], dtype=torch.long) for item in batch]
    attention_mask = [torch.tensor(item["attention_mask"], dtype=torch.long) for item in batch]
    labels = [torch.tensor(item["label"], dtype=torch.long) for item in batch]
    
    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=0)
    attention_mask = torch.nn.utils.rnn.pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels = torch.stack(labels)
    
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "label": labels
    }

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=custom_collate_fn)
dev_loader = DataLoader(dev_dataset, batch_size=4, shuffle=False, collate_fn=custom_collate_fn)


config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

[transformers] You are using a model of type `modern_decoder` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


tokenizer_config.json:   0%|          | 0.00/223 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.28M [00:00<?, ?B/s]

Processing Dataset Features...


In [4]:
# ==========================================
# 3. GQA BITFIT ARCHITECTURE DEFINITION
# ==========================================
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        # RMSNorm weights are part of normalization tuning parameters
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        variance = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(variance + self.eps) * self.weight

class BitFitLinear(nn.Module):
    """
    Standard Linear layer configured for BitFit:
    Weights are frozen (requires_grad=False), biases are trainable (requires_grad=True).
    """
    def __init__(self, in_features: int, out_features: int, bias: bool = True):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_features, in_features), requires_grad=False)
        self.bias = nn.Parameter(torch.zeros(out_features), requires_grad=True) if bias else None
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.linear(x, self.weight, self.bias)

class GroupedQueryAttentionBitFit(nn.Module):
    def __init__(self, d_model: int, num_query_heads: int, num_kv_heads: int):
        super().__init__()
        self.d_model = d_model
        self.num_query_heads = num_query_heads
        self.num_kv_heads = num_kv_heads
        
        assert num_query_heads % num_kv_heads == 0
        self.num_queries_per_kv = num_query_heads // num_kv_heads
        self.head_dim = d_model // num_query_heads
        
        self.q_proj = BitFitLinear(d_model, d_model)
        self.k_proj = BitFitLinear(d_model, num_kv_heads * self.head_dim)
        self.v_proj = BitFitLinear(d_model, num_kv_heads * self.head_dim)
        self.out_proj = BitFitLinear(d_model, d_model)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        batch_size, seq_len, _ = x.shape
        q = self.q_proj(x).view(batch_size, seq_len, self.num_query_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
        
        if self.num_queries_per_kv > 1:
            k = k.repeat_interleave(self.num_queries_per_kv, dim=1)
            v = v.repeat_interleave(self.num_queries_per_kv, dim=1)
            
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-10000.0'))
            
        attn_weights = F.softmax(scores, dim=-1)
        context = torch.matmul(attn_weights, v)
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.out_proj(context)

class SwiGLUFFNBitFit(nn.Module):
    def __init__(self, d_model: int, hidden_dim: int):
        super().__init__()
        self.gate_proj = BitFitLinear(d_model, hidden_dim)
        self.up_proj = BitFitLinear(d_model, hidden_dim)
        self.down_proj = BitFitLinear(hidden_dim, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

class ModernDecoderBlockBitFit(nn.Module):
    def __init__(self, d_model: int, num_query_heads: int, num_kv_heads: int):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = GroupedQueryAttentionBitFit(d_model, num_query_heads, num_kv_heads)
        self.norm2 = RMSNorm(d_model)
        hidden_dim = int(8 * d_model / 3)
        self.ffn = SwiGLUFFNBitFit(d_model, hidden_dim)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        x = x + self.attn(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x

class GQABitFitClassifier(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, num_layers: int, num_query_heads: int, num_kv_heads: int, num_labels: int, max_len: int = 1024):
        super().__init__()
        # Freeze token and positional embeddings for BitFit
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.token_embedding.weight.requires_grad = False
        self.pos_embedding = nn.Embedding(max_len, d_model)
        self.pos_embedding.weight.requires_grad = False
        
        self.blocks = nn.ModuleList([
            ModernDecoderBlockBitFit(d_model, num_query_heads, num_kv_heads) 
            for _ in range(num_layers)
        ])
        
        self.norm_f = RMSNorm(d_model)
        self.classifier = nn.Linear(d_model, num_labels) # Fully trainable classification head

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor = None) -> torch.Tensor:
        batch_size, seq_len = input_ids.shape
        if seq_len > self.pos_embedding.num_embeddings:
            input_ids = input_ids[:, :self.pos_embedding.num_embeddings]
            if attention_mask is not None:
                attention_mask = attention_mask[:, :self.pos_embedding.num_embeddings]
            seq_len = self.pos_embedding.num_embeddings

        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, seq_len)
        x = self.token_embedding(input_ids) + self.pos_embedding(positions)
        
        mask = None
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(1).unsqueeze(2)
            
        for block in self.blocks:
            x = block(x, mask)
            
        x = self.norm_f(x)
        
        if attention_mask is not None:
            mask_expanded = attention_mask.unsqueeze(-1).float()
            sum_embeddings = torch.sum(x * mask_expanded, dim=1)
            sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
            pooled = sum_embeddings / sum_mask
        else:
            pooled = x.mean(dim=1)
            
        logits = self.classifier(pooled)
        return logits

In [5]:
# ==========================================
# 4. MODEL INITIALIZATION & PHASE 1 WEIGHT LOADING (SAFETENSORS COMPLIANT)
# ==========================================
import safetensors.torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VOCAB_SIZE = len(tokenizer)

HF_MODEL_REPO = "A-240242/legal-gqa-checkpoints"

print("Initializing GQA BitFit Classifier with Phase 1 dimensions...")
model = GQABitFitClassifier(
    vocab_size=VOCAB_SIZE, 
    d_model=768,        # Matches checkpoint hidden dimension
    num_layers=6,        # Adjust to match your Phase 1 config (commonly 6 or 12 for d_model=768)
    num_query_heads=12,  # Adjust to match your Phase 1 head config
    num_kv_heads=4,      # Adjust to match your Phase 1 GQA KV heads config
    num_labels=2,
    max_len=2048         # Matches checkpoint max length
).to(device)

print("Downloading and loading Phase 1 pre-trained weights via safetensors...")
try:
    weights_path = hf_hub_download(
        repo_id=HF_MODEL_REPO,
        filename="model.safetensors",
        token=HF_TOKEN
    )
    # Load state dict correctly using safetensors instead of torch.load
    state_dict = safetensors.torch.load_file(weights_path, device=str(device))
    model.load_state_dict(state_dict, strict=False)
    print("Phase 1 GQA weights loaded successfully with safetensors!")
except Exception as e:
    print(f"Notice: Could not load pre-trained weights ({e}). Initializing with structural layers.")

# Enforce BitFit constraint: Only optimize parameters where requires_grad=True (biases, norms, classifier)
trainable_params = [p for p in model.parameters() if p.requires_grad]
print(f"Total Trainable Parameters under BitFit: {sum(p.numel() for p in trainable_params):,}")

optimizer = torch.optim.AdamW(trainable_params, lr=1e-3)
criterion = nn.CrossEntropyLoss()

EPOCHS = 3
print("\nStarting GQA BitFit Training Loop...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{EPOCHS} | BitFit Train Loss: {total_loss / len(train_loader):.4f}")
    
    # Validation evaluation
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.inference_mode():
        for batch in dev_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            
            logits = model(input_ids, attention_mask)
            preds = logits.argmax(dim=-1).cpu().tolist()
            
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().tolist())
            
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)
    print(f"Validation Macro-F1: {f1:.4f} (Precision: {precision:.4f}, Recall: {recall:.4f})")

print("\nGQA BitFit training pipeline complete!")

Initializing GQA BitFit Classifier with Phase 1 dimensions...


model.safetensors: reconstructing file:   0%|          |  0.00B /  354MB            

model.safetensors: downloading bytes:           |  0.00B            

Phase 1 GQA weights loaded successfully with safetensors!
Total Trainable Parameters under BitFit: 52,994

Starting GQA BitFit Training Loop...
Epoch 1/3 | BitFit Train Loss: 0.6806
Validation Macro-F1: 0.3336 (Precision: 0.2503, Recall: 0.5000)
Epoch 2/3 | BitFit Train Loss: 0.6780
Validation Macro-F1: 0.3336 (Precision: 0.2503, Recall: 0.5000)
Epoch 3/3 | BitFit Train Loss: 0.6745
Validation Macro-F1: 0.4650 (Precision: 0.5704, Recall: 0.5340)

GQA BitFit training pipeline complete!


In [6]:
# ==========================================
# 5. INFERENCE & SUBMISSION GENERATION (GQA BITFIT)
# ==========================================
print("Loading Test Data for Submission Generation...")
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO, 
    filename="Phase_4_CJPE/Test/test.json", 
    repo_type="dataset", 
    token=HF_TOKEN
)

with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

model.eval()
predictions = []

print(f"Running inference on {len(test_data)} test examples...")
for case in test_data:
    case_id = case.get("id")
    case_text = case.get("text", "")
    
    # Tokenize input test text
    encoded = tokenizer(
        case_text, 
        truncation=False, 
        return_attention_mask=False
    )
    full_ids = encoded["input_ids"]
    
    # Apply the same 1024 token truncation used during training
    chunk = full_ids[:1024]
    
    input_ids = torch.tensor([chunk], dtype=torch.long).to(device)
    attention_mask = torch.tensor([[1] * len(chunk)], dtype=torch.long).to(device)
    
    with torch.inference_mode():
        # Get logits from the GQA BitFit model forward pass
        logits = model(input_ids, attention_mask)
        pred_label = int(logits.argmax(dim=-1).item())
        
    predictions.append({
        "id": case_id,
        "pred": pred_label
    })

# Save to expected submission format (`prediction.json` zipped)
output_dir = "./phase4_gqa_bitfit_cjpe"
os.makedirs(output_dir, exist_ok=True)
pred_path = os.path.join(output_dir, "prediction.json")
sub_zip_path = os.path.join(output_dir, "submission.zip")

with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(sub_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(pred_path, arcname="prediction.json")

print(f"\nCJPE Prediction complete! Generated {len(predictions)} predictions.")
print(f"Submission successfully saved to: {sub_zip_path}")

Loading Test Data for Submission Generation...


Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

Running inference on 1517 test examples...

CJPE Prediction complete! Generated 1517 predictions.
Submission successfully saved to: ./phase4_gqa_bitfit_cjpe/submission.zip
